# 03 — The Breakthrough Catalog

**Ground truth dataset of known technological breakthroughs.**

This notebook loads our curated catalog of 34 breakthroughs, validates them
against the patent database, and analyzes their citation structure. This
establishes the ground truth for the hypothesis test in Notebook 04.

---

In [1]:
# %% Imports and setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import DATA_DIR, get_logger
from src.breakthroughs import load_breakthroughs, get_cpc_context, get_citation_context, get_precursor_window
from src.plotting import set_style, save_figure, PALETTE, PALETTE_LIST, CPC_COLORS, CPC_LABELS, year_axis

set_style()
logger = get_logger('nb03')

In [2]:
# %% Load data
patents = pd.read_parquet(DATA_DIR / 'patents.parquet')
citations = pd.read_parquet(DATA_DIR / 'citations.parquet')
cpc_map = pd.read_parquet(DATA_DIR / 'cpc_map.parquet')

breakthroughs = load_breakthroughs()
print(f'Loaded {len(breakthroughs)} breakthroughs')

2026-03-19 20:24:59 | src.breakthroughs        | INFO    | Loaded 34 breakthroughs from 8 files


Loaded 34 breakthroughs


## Validate Catalog Against Patent Database

Do the breakthrough patents exist in our dataset? Which ones are missing?

In [3]:
# %% Validate breakthroughs
valid_ids = set(patents['patent_id'])
validation = []

for bt in breakthroughs:
    ctx = get_cpc_context(bt, cpc_map)
    found = len(ctx['found_patents'])
    total = len(bt.breakthrough_patents)
    
    validation.append({
        'name': bt.name,
        'category': bt.category,
        'filing_year': bt.filing_year,
        'patents_in_catalog': total,
        'patents_found': found,
        'patents_missing': total - found,
        'cpc_sections_declared': ','.join(bt.cpc_sections),
        'cpc_sections_found': ','.join(sorted(ctx['cpc_sections'])),
    })

val_df = pd.DataFrame(validation)
print(f'\nBreakthroughs with all patents found: {(val_df["patents_missing"] == 0).sum()} / {len(val_df)}')
print(f'Total patents checked: {val_df["patents_in_catalog"].sum()}')
print(f'Total patents found: {val_df["patents_found"].sum()}')

if (val_df['patents_missing'] > 0).any():
    print('\nBreakthroughs with missing patents:')
    missing = val_df[val_df['patents_missing'] > 0]
    for _, row in missing.iterrows():
        print(f"  {row['name']}: {row['patents_missing']}/{row['patents_in_catalog']} missing")


Breakthroughs with all patents found: 34 / 34
Total patents checked: 67
Total patents found: 67


## Figure 1: Timeline of Cataloged Breakthroughs

In [4]:
# %% Figure 1: Breakthrough timeline
category_colors = {
    'biotech': PALETTE['green'],
    'computing': PALETTE['blue'],
    'materials': PALETTE['orange'],
    'energy': PALETTE['red'],
    'telecom': PALETTE['cyan'],
    'manufacturing': PALETTE['purple'],
    'ai_ml': PALETTE['yellow'],
    'security': PALETTE['black'],
}

fig, ax = plt.subplots(figsize=(16, 8))

categories = sorted(set(bt.category for bt in breakthroughs))
cat_y = {cat: i for i, cat in enumerate(categories)}

for bt in breakthroughs:
    y = cat_y[bt.category] + np.random.uniform(-0.2, 0.2)  # jitter
    ax.scatter(bt.filing_year, y, s=80, c=category_colors.get(bt.category, 'gray'),
               edgecolors='white', linewidth=0.5, zorder=5)
    ax.annotate(bt.name, (bt.filing_year, y), fontsize=6,
                xytext=(5, 3), textcoords='offset points', alpha=0.8)

ax.set_yticks(range(len(categories)))
ax.set_yticklabels([cat.replace('_', '/').title() for cat in categories])
ax.set_xlabel('Filing Year')
ax.set_title('Figure 1: Timeline of Cataloged Technological Breakthroughs')
year_axis(ax, 1975, 2020)
fig.tight_layout()
save_figure(fig, '03_breakthrough_timeline')

2026-03-19 20:27:56 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_breakthrough_timeline.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_breakthrough_timeline.png')

## Figure 2: Citation Fan-Out

How quickly do breakthrough patents accumulate citations?

In [5]:
# %% Figure 2: Citation fan-out for selected breakthroughs
selected = [bt for bt in breakthroughs if bt.name in [
    'CRISPR-Cas9 Gene Editing', 'PageRank Algorithm', 'Lithium-Ion Battery',
    'Polymerase Chain Reaction (PCR)', 'Multi-Touch Interface (iPhone)',
    'Selective Laser Sintering (SLS) 3D Printing',
]]

fig, ax = plt.subplots(figsize=(12, 7))

for i, bt in enumerate(selected):
    bt_ids = set(bt.breakthrough_patents)
    fwd = citations[citations['cited_id'].isin(bt_ids)].copy()
    fwd['year'] = pd.to_datetime(fwd['citing_date']).dt.year
    
    if len(fwd) == 0:
        continue
    
    annual = fwd.groupby('year').size().sort_index().cumsum()
    ax.plot(annual.index, annual.values, linewidth=2, label=bt.name,
            color=PALETTE_LIST[i % len(PALETTE_LIST)])

ax.set_xlabel('Year')
ax.set_ylabel('Cumulative Forward Citations')
ax.set_title('Figure 2: Citation Fan-Out for Selected Breakthroughs')
ax.legend(fontsize=9)
fig.tight_layout()
save_figure(fig, '03_citation_fanout')

2026-03-19 20:29:43 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_citation_fanout.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_citation_fanout.png')

## Figure 3: CPC Section Map

Which section pairs are involved in each breakthrough?

In [6]:
# %% Figure 3: Breakthrough-to-CPC mapping
sections = list('ABCDEFGH')
matrix = np.zeros((len(breakthroughs), len(sections)))

for i, bt in enumerate(breakthroughs):
    for sec in bt.cpc_sections:
        if sec in sections:
            matrix[i, sections.index(sec)] = 1

fig, ax = plt.subplots(figsize=(10, 12))
ax.imshow(matrix, aspect='auto', cmap='Blues', interpolation='nearest')
ax.set_xticks(range(len(sections)))
ax.set_xticklabels(sections)
ax.set_yticks(range(len(breakthroughs)))
ax.set_yticklabels([bt.name for bt in breakthroughs], fontsize=7)
ax.set_xlabel('CPC Section')
ax.set_title('Figure 3: Breakthrough × CPC Section Involvement')
fig.tight_layout()
save_figure(fig, '03_cpc_section_map')

2026-03-19 20:29:46 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_cpc_section_map.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_cpc_section_map.png')

## Figure 4: Precursor Analysis

How many of each breakthrough's cited references come from outside
its primary CPC section? Cross-disciplinary precursors may signal
that the breakthrough drew on diverse knowledge.

In [7]:
# %% Figure 4: Cross-section precursor fraction
precursor_data = []

primary_section = cpc_map.groupby('patent_id')['cpc_section'].first()

for bt in breakthroughs:
    bt_ids = set(bt.breakthrough_patents)
    # Backward citations: patents cited BY the breakthrough
    bwd = citations[citations['citing_id'].isin(bt_ids)]
    
    if len(bwd) == 0:
        continue
    
    # Get sections of cited patents
    cited_sections = bwd['cited_id'].map(primary_section).dropna()
    primary_sec = bt.cpc_sections[0] if bt.cpc_sections else None
    
    if primary_sec and len(cited_sections) > 0:
        cross_frac = (cited_sections != primary_sec).mean()
        precursor_data.append({
            'name': bt.name,
            'category': bt.category,
            'cross_section_fraction': cross_frac,
            'n_precursors': len(cited_sections),
        })

prec_df = pd.DataFrame(precursor_data).sort_values('cross_section_fraction', ascending=False)

fig, ax = plt.subplots(figsize=(12, 8))
colors = [category_colors.get(row['category'], 'gray') for _, row in prec_df.iterrows()]
ax.barh(range(len(prec_df)), prec_df['cross_section_fraction'], color=colors, edgecolor='white')
ax.set_yticks(range(len(prec_df)))
ax.set_yticklabels(prec_df['name'], fontsize=8)
ax.set_xlabel('Fraction of Precursor Patents from Different CPC Section')
ax.set_title('Figure 4: Cross-Disciplinary Precursor Fraction per Breakthrough')
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)
ax.invert_yaxis()
fig.tight_layout()
save_figure(fig, '03_precursor_cross_section')

2026-03-19 20:32:55 | src.plotting             | INFO    | Saved figure: /Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_precursor_cross_section.png


PosixPath('/Users/christopherortiz/Desktop/the-shape-of-discovery/figures/03_precursor_cross_section.png')

## Summary

The breakthrough catalog is validated. Key findings:
1. **Coverage** — How many breakthrough patents exist in our dataset
2. **Citation impact** — Fan-out curves show the influence trajectory
3. **Cross-disciplinary origins** — Which breakthroughs drew most heavily on knowledge outside their primary field

This ground truth feeds directly into Notebook 04's hypothesis test.